<a href="https://colab.research.google.com/github/peremartra/Rearchitecting-LLMs/blob/main/APPD/APPD_NB01_examples.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<table style="width:100%; border:none; background:none;">
  <tr style="border:none;">
    <td style="border:none; vertical-align:middle; text-align:left; width: 120px;">
      <a href="https://hubs.la/Q040tvsK0"><img src="https://raw.githubusercontent.com/peremartra/Rearchitecting-LLMs/main/Images/cover.png" width="100px" style="border-radius: 4px;"></a>
    </td>
    <td style="border:none; vertical-align:middle; text-align:left;">
      <p style="margin: 0; font-size: 14px;">
        Supplementary code for the <a href="https://hubs.la/Q040tvsK0">Rearchitecting LLMs</a> book by <a href="https://github.com/peremartra">Pere Martra</a>.<br>
        <br>
        Code repository: <a href="https://github.com/peremartra/Rearchitecting-LLMs">https://github.com/peremartra/Rearchitecting-LLMs</a>
      </p>
    </td>
  </tr>
</table>

# Rearchitecting LLMs
## Structural techniques for efficient models


### Appendix D: Inference performance evaluation
### Notebook: 01. How to measure inference performance

[![LinkedIn](https://img.shields.io/badge/LinkedIn-0077B5?style=flat&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/pere-martra/) [![GitHub](https://img.shields.io/badge/GitHub-100000?style=flat&logo=github&logoColor=white)](https://github.com/peremartra) [![X](https://img.shields.io/badge/X-000000?style=flat&logo=x&logoColor=white)](https://x.com/PereMartra) [![Hugging Face](https://img.shields.io/badge/🤗%20Hugging%20Face-blue)](https://huggingface.co/oopere)

_____
Colab Environment: GPU T4

Models:
* qwen3-0.6b-weather-geo-specialist
* qwen3-0.6b-weather-geo-S
_____

## Setting up the notebook

In [1]:
# Install dependencies
!pip install -q "transformers==5.18.0" "accelerate==1.15.0" datasets pandas

In [2]:
import gc, time
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer


assert torch.cuda.is_available(), "This notebook needs a GPU runtime"
print(f"GPU: {torch.cuda.get_device_name(0)}")

GPU: Tesla T4


In [3]:
# Models: the fine-tuned specialist and the pruned and distilled version of it
SPECIALIST = "oopere/qwen3-0.6b-weather-geo-specialist"
PRUNED = "oopere/qwen3-0.6b-weather-geo-S"

# Measurement configuration
PROMPT_LENGTHS = [256, 1024, 2048]  # real tokens in every prompt
BATCH_SIZE = 4
NUM_BATCHES = 3                     # 12 prompts per length
NUM_RUNS = 3                        # repetitions of every batch
MAX_NEW_TOKENS = 50

In [4]:
def clear_gpu_cache():
    """Clear GPU cache completely"""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    gc.collect()

## Preparing the text

The speed of a model does not depend on what the prompt says, only on its length. We take WikiText only as a source of real tokens to cut prompts of an exact length.

In [5]:
tokenizer = AutoTokenizer.from_pretrained(SPECIALIST)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

wiki = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train[:3000]")
text = "\n\n".join(t for t in wiki["text"] if t.strip())
token_ids = tokenizer(text, add_special_tokens=False, return_tensors="pt")["input_ids"][0]
print(f"{len(token_ids):,} tokens available")

config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/4.17k [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (207559 > 131072). Running this sequence through the model will result in indexing errors


207,559 tokens available


### Listing D.1 Building prompts of exact length

In [6]:
def build_prompts(token_ids, prompt_length, batch_size=4, num_batches=3):
    num_prompts = batch_size * num_batches
    needed = num_prompts * prompt_length
    assert len(token_ids) >= needed, "Not enough text for this length"  #A

    ids = token_ids[:needed].view(num_prompts, prompt_length)  #B

    return [
        {
            "input_ids": ids[i:i + batch_size],
            "attention_mask": torch.ones(batch_size, prompt_length,
                                         dtype=torch.long)  #C
        }
        for i in range(0, num_prompts, batch_size)
    ]

In [7]:
batches_by_length = {
    length: build_prompts(token_ids, length, BATCH_SIZE, NUM_BATCHES)
    for length in PROMPT_LENGTHS
}
print({length: tuple(b[0]["input_ids"].shape) for length, b in batches_by_length.items()})

{256: (4, 256), 1024: (4, 1024), 2048: (4, 2048)}


### Listing D.2 Measuring latency and throughput

In [8]:
def measure_inference(model, tokenizer, batches, num_runs=3, max_new_tokens=50):
    model.eval()
    device = model.device

    def generate(batch):
        return model.generate(
            batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device),
            max_new_tokens=max_new_tokens,
            min_new_tokens=max_new_tokens,  #A
            do_sample=False,
            temperature=None, top_p=None, top_k=None,
            pad_token_id=tokenizer.pad_token_id
        )

    with torch.no_grad():
        for _ in range(2):
            generate(batches[0])  #B
        torch.cuda.synchronize()

        latencies, total_tokens = [], 0
        for batch in batches:
            batch_size, prompt_length = batch["input_ids"].shape
            for _ in range(num_runs):
                torch.cuda.synchronize()  #C
                start = time.perf_counter()
                outputs = generate(batch)
                torch.cuda.synchronize()
                latencies.append(time.perf_counter() - start)

                new_tokens = outputs.shape[1] - prompt_length
                total_tokens += new_tokens * batch_size  #D

    return {
        "avg_latency_sec": float(np.mean(latencies)),
        "std_latency_sec": float(np.std(latencies)),
        "throughput_tokens_per_sec": total_tokens / sum(latencies),  #E
        "total_tokens": total_tokens,
        "dtype_used": str(next(model.parameters()).dtype)  #F
    }

## Checking the models before measuring

The pruned model is stored in bfloat16, which a T4 only emulates. To measure in a precision the GPU supports natively, both models are loaded in float16. Before measuring anything we check that the conversion has not broken the model.

In [9]:
# A request from the models' own domain: both are tool-calling specialists,
# so a general-knowledge prompt says nothing about whether fp16 broke them.
SANITY_TOOLS = [{
    "name": "get_city_from_zipcode",
    "description": "Retrieves the city name for a given ZIP code.",
    "parameters": {"zipcode": {"description": "The ZIP code to look up.",
                               "type": "str", "default": ""}},
}]
SANITY_PROMPT = tokenizer.apply_chat_template(
    [{"role": "user", "content": "What city is associated with the ZIP code 94117?"}],
    tools=SANITY_TOOLS, tokenize=False, add_generation_prompt=True,
    enable_thinking=False)

def sanity_check(model_id, prompt=SANITY_PROMPT):
    model = AutoModelForCausalLM.from_pretrained(
        model_id, dtype=torch.float16, device_map="auto")
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits
        output = model.generate(**inputs, max_new_tokens=40, do_sample=False,
                                temperature=None, top_p=None, top_k=None)
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    print(f"{model_id.split('/')[-1]} | finite logits: {bool(torch.isfinite(logits).all())}")
    print("  " + tokenizer.decode(new_tokens, skip_special_tokens=True))
    del model
    clear_gpu_cache()

## Measuring one model

In [10]:
model = AutoModelForCausalLM.from_pretrained(
    SPECIALIST, dtype=torch.float16, device_map="auto")

result = measure_inference(model, tokenizer, batches_by_length[1024],
                           num_runs=NUM_RUNS, max_new_tokens=MAX_NEW_TOKENS)
print(result)
print("Attention implementation:", model.config._attn_implementation)
del model
clear_gpu_cache()


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

{'avg_latency_sec': 3.7817341651111116, 'std_latency_sec': 0.8738079670002774, 'throughput_tokens_per_sec': 52.88579029301595, 'total_tokens': 1800, 'dtype_used': 'torch.float16'}
Attention implementation: sdpa


### Listing D.3 Sweeping prompt lengths with a noise control

In [11]:
def run_sweep(variants, batches_by_length, num_runs=3, max_new_tokens=50):
    rows = []
    for name, model_id in variants:
        baseline = torch.cuda.memory_allocated()
        model = AutoModelForCausalLM.from_pretrained(
            model_id, dtype=torch.float16, device_map="auto")  #A
        params = sum(p.numel() for p in model.parameters())
        weights_gb = sum(p.numel() * p.element_size()
                         for p in model.parameters()) / 1024**3

        for length, batches in batches_by_length.items():
            torch.cuda.reset_peak_memory_stats()
            first = measure_inference(model, tokenizer, batches,
                                      num_runs, max_new_tokens=1)  #B
            full = measure_inference(model, tokenizer, batches,
                                     num_runs, max_new_tokens=max_new_tokens)
            peak_gb = (torch.cuda.max_memory_allocated() - baseline) / 1024**3

            batch_size = batches[0]["input_ids"].shape[0]
            decode_sec = full["avg_latency_sec"] - first["avg_latency_sec"]
            rows.append({
                "model": name,
                "layers": model.config.num_hidden_layers,
                "mlp_size": model.config.intermediate_size,
                "params_M": params / 1e6,
                "L": length,
                "prefill_ms": first["avg_latency_sec"] * 1000,
                "tok_s": full["throughput_tokens_per_sec"],
                "decode_tok_s": batch_size * (max_new_tokens - 1) / decode_sec,
                "weights_GB": weights_gb,
                "peak_GB": peak_gb
            })
        print(f"done: {name}")
        del model
        clear_gpu_cache()  #C
    return pd.DataFrame(rows)

In [12]:
variants = [
    ("Specialist (warm-up)", SPECIALIST),  # warms up the GPU, never reported
    ("Specialist", SPECIALIST),
    ("S", PRUNED),
    ("Specialist (again)", SPECIALIST),    # same model again, measures the noise
]

sweep = run_sweep(variants, batches_by_length, NUM_RUNS, MAX_NEW_TOKENS)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

done: Specialist (warm-up)


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

done: Specialist


config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/244 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

done: S


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

done: Specialist (again)


## Results

In [13]:
order = [name for name, _ in variants if "warm-up" not in name]

def table(column):
    return sweep.pivot(index="model", columns="L", values=column).reindex(order)

prefill = table("prefill_ms")
speed = table("tok_s")

print("Time to generate the first token, batch of 4 (ms). Lower is better")
display(prefill.round(0))
print("Tokens per second, 50 new tokens. Higher is better")
display(speed.round(1))
print("Tokens per second vs. Specialist (%)")
display(((speed / speed.loc["Specialist"] - 1) * 100).round(1))

decode = table("decode_tok_s")
print("Decode only: tokens per second after the first token. Higher is better")
display(decode.round(1))
print("Decode tokens per second vs. Specialist (%)")
display(((decode / decode.loc["Specialist"] - 1) * 100).round(1))

longest = PROMPT_LENGTHS[-1]
print(f"Model size and GPU memory (GB), peak measured at L={longest}")
display(sweep[sweep.L == longest].set_index("model").reindex(order)
        [["layers", "mlp_size", "params_M", "weights_GB", "peak_GB"]].round(2))

Time to generate the first token, batch of 4 (ms). Lower is better


L,256,1024,2048
model,,,
Specialist,143.0,1001.0,3121.0
S,100.0,786.0,2342.0
Specialist (again),142.0,1012.0,3119.0


Tokens per second, 50 new tokens. Higher is better


L,256,1024,2048
model,,,
Specialist,75.2,55.5,26.5
S,96.9,71.1,34.1
Specialist (again),72.0,54.4,26.5


Tokens per second vs. Specialist (%)


...

L,256,1024,2048
model,,,
Specialist,77.9,75.3,44.3
S,99.8,96.7,55.7
Specialist (again),74.4,73.6,44.2


Decode tokens per second vs. Specialist (%)


L,256,1024,2048
model,,,
Specialist,0.0,0.0,0.0
S,28.2,28.4,25.8
Specialist (again),-4.5,-2.2,-0.2


Model size and GPU memory (GB), peak measured at L=2048


,layers,mlp_size,params_M,weights_GB,peak_GB
model,,,,,
Specialist,28,3072,596.05,1.11,4.65
S,22,1792,415.16,0.77,4.12
Specialist (again),28,3072,596.05,1.11,4.65


### Noise floor
The two measurements of the same model show how much a result can move without any change in the model. A difference is only worth interpreting when it is larger than that.

In [14]:
noise = ((speed.loc["Specialist (again)"] / speed.loc["Specialist"] - 1) * 100).abs()
gain = (speed.loc["S"] / speed.loc["Specialist"] - 1) * 100

verdict = pd.DataFrame({
    "noise floor (%)": noise.round(1),
    "S vs. Specialist (%)": gain.round(1),
    "beyond the noise": gain.abs() > noise
})
display(verdict)

,noise floor (%),S vs. Specialist (%),beyond the noise
L,,,
256,4.2,28.9,True
1024,1.9,28.1,True
2048,0.1,28.8,True


## Memory taken by the KV cache

The peak memory also includes the KV cache, which grows with the prompt length. It can be computed from the configuration, and depth pruning reduces it because it removes layers.

In [15]:
for name, model_id in (("Specialist", SPECIALIST), ("S", PRUNED)):
    cfg = AutoConfig.from_pretrained(model_id)
    per_token = 2 * cfg.num_hidden_layers * cfg.num_key_value_heads * cfg.head_dim * 2  # K and V, fp16
    total_gb = per_token * longest * BATCH_SIZE / 1024**3
    print(f"{name}: {per_token / 1024:.0f} KB per token, "
          f"{total_gb:.2f} GB for {BATCH_SIZE} prompts of {longest} tokens")

Specialist: 112 KB per token, 0.88 GB for 4 prompts of 2048 tokens
S: 88 KB per token, 0.69 GB for 4 prompts of 2048 tokens
